# Man-in-the-Middle — Passthrough Proxy

An attacker has man-in-the-middled the link between the GCS and the vehicle's
onboard Logic. The MITM sits as a transparent proxy:

```
telemetry:  Logic --MITM_TELEM--> [MITM] --GCS--> GCS
commands:   GCS   --MITM_CMD-->   [MITM] --GCS_CMD--> Logic
```

With the default `passthrough` strategy every message is forwarded unmodified,
so the scenario behaves *identically* to the no-MITM case: the drone flies the
north AUTO mission and the GCS intervention still redirects it east. This is the
baseline that future attack strategies (drop / modify / inject) build on.

Toggle the attack off by setting `simulator.mitm = None`.

In [ ]:
from simulator import Simulator
from simulator.config import PARAMS_PATH, Color, Model
from simulator.entities import SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin and waypoints

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, 0, 0, 0)
speed = 5.0    # m/s
cruise_alt = 10.0  # m
model = Model.IRIS
sysid = 1

# Mission seq: seq=0 home, seq=1 TAKEOFF, seq=2 flying to north_100,
#              seq=3 flying to north_200 ← intervention fires here
home_wp    = ENU(x=0, y=0,   z=0)
north_100  = ENU(x=0, y=100, z=cruise_alt)
north_200  = ENU(x=0, y=200, z=cruise_alt)
mission_wps = [home_wp, north_100, north_200]

# Intervention target: 100 m east of origin at cruise altitude
east_target = gra_origin.unpose().to_abs(ENU(x=100, y=0, z=cruise_alt))
print(f"Intervention target: lat={east_target.lat:.7f}, lon={east_target.lon:.7f}")

## Vehicle

In [ ]:
mission_path = "simulator/planner/missions/mitm_north.waypoints"

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=mission_path,
    navigation_speed=speed,
    firmware=model.firmware,
)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    gcs_name=f"BLUE_{Color.BLUE.emoji}",
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
)

## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_marker)

## Simulator + MITM

In [ ]:
simulator = Simulator(visualizer=gaz, verbose=1)
simulator.add_vehicle(vehicle, parm=str(PARAMS_PATH / "vehicle.parm"))

# GCS intervention: when MISSION_CURRENT.seq >= 3 (flying north_100 -> north_200),
# switch to GUIDED and reposition east. `simulator.intervention` is keyed by
# sysid, so this targets only this vehicle.
simulator.intervention[sysid] = {
    "trigger_seq": 3,
    "target_lat": east_target.lat,
    "target_lon": east_target.lon,
    "target_alt": cruise_alt,
}

# Interpose the man-in-the-middle proxy on this vehicle only. "passthrough"
# forwards every message unmodified, so the run should look identical to the
# no-MITM sanity check. `simulator.mitm` is keyed by sysid, so other vehicles
# sharing this GCS (if any) would be unaffected unless also added here.
simulator.mitm[sysid] = {"strategy": "passthrough"}

simulator.show()

In [ ]:
orac = simulator.launch()
orac.run()

## Verifying the MITM was in the path

Logs to check:

- `simulator/logs/mitm/mitm_1.log` — `MITM proxy active for vehicle 1 (strategy=PassthroughStrategy)`
- `simulator/logs/GCSs/GCS_BLUE_*.log` — `GCS intervention: switching vehicle 1 ...`
- `simulator/logs/logics/logic_1.log` — `GCS→SITL forwarding SET_MODE` / `COMMAND_INT`

If the intervention still fires and the drone turns east, telemetry and commands
both traversed the MITM transparently.